# Slowly Changing Dimension (Type 2) Example using Jinja2

Generates the two SCD2 statements -- expire-then-insert -- from a Jinja2 template, so the
tracked columns and keys are declared once instead of hand-written per dimension.

## Prerequisites

1. **jinja2** on the cluster. Install it from the `requirements.txt` next to this
   notebook (Cluster -> Libraries -> install from requirements.txt).
2. **A catalog you can write to.** Set `CATALOG` in the configuration cell below;
   the notebook creates the catalog and schema if they do not exist.

## Preconditions on the staging data

The MERGE expires **one** current row per key, so the staging table must hold **at most
one row per SCD key**. Duplicate keys make the MERGE fail with a multiple-match error and
would otherwise produce two current rows for the same key. De-duplicate upstream.

See the jinja2 docs for more examples: https://jinja.palletsprojects.com/en/stable/

In [ ]:
# Configuration -- point this at a catalog you can write to.
# jinja2 comes from requirements.txt (see Prerequisites); no %pip install needed.

CATALOG = "default"          # change to your own catalog if required
SCHEMA = "scd"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.dim_customer"
SOURCE_VIEW = "staging_customer"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"Target table: {TARGET_TABLE}")

In [ ]:
# Clean up any previous run. Runs after the CREATE above, so the schema exists.
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

## The Jinja2 template for handling slowly changing behavior

In [ ]:
from jinja2 import Template

# Change detection uses the NULL-safe equality operator <=> rather than !=.
# With !=, a NULL on either side yields NULL, which is falsy in both WHEN MATCHED AND
# and WHERE -- so a NULL -> value transition (or value -> NULL) is silently dropped:
# the row is never expired and never re-inserted. NOT (a <=> b) treats NULL as a value
# and compares correctly. Spark also accepts `IS DISTINCT FROM` with identical semantics.

scd2_template_update = Template("""

-- Step 1: Expire old records
MERGE INTO {{ target_table }} AS target
USING {{ source_table }} AS source
ON {{ scd_keys }}
AND target.current_flag = true
WHEN MATCHED AND (
    {% for col in tracked_columns %}
        NOT (target.{{ col }} <=> source.{{ col }}){% if not loop.last %} OR {% endif %}
    {% endfor %}
)
THEN UPDATE SET
    current_flag = false,
    effective_end_date = current_date();
""")

scd2_template_insert = Template("""

-- Step 2: Insert new/changed records
INSERT INTO {{ target_table }} (
    {{ insert_columns | join(', ') }},
    effective_start_date,
    effective_end_date,
    current_flag
)
SELECT
    {% for col in insert_columns %}
        source.{{ col }}{% if not loop.last %}, {% endif %}
    {% endfor %},
    current_date(),
    NULL,
    true
FROM {{ source_table }} AS source
LEFT JOIN {{ target_table }} AS target
ON {{ scd_keys }}
AND target.current_flag = true
WHERE
    target.{{ key_column }} IS NULL OR
    {% for col in tracked_columns %}
        NOT (target.{{ col }} <=> source.{{ col }}){% if not loop.last %} OR {% endif %}
    {% endfor %};

""")


def run_scd2_merge(source_table, target_table, scd_keys, key_column,
                   tracked_columns, insert_columns, verbose=False):
    """Run the SCD2 expire-then-insert pair.

    key_column -- the target's key column, used for the "no current row" test in
    step 2. Passed explicitly so the template is not tied to one dimension.
    """
    params = dict(source_table=source_table, target_table=target_table,
                  scd_keys=scd_keys, key_column=key_column,
                  tracked_columns=tracked_columns, insert_columns=insert_columns)

    for label, template in (("Update", scd2_template_update),
                            ("Insert", scd2_template_insert)):
        sql = template.render(**params)
        if verbose:
            print(f"Executing {label} SQL:\n", sql)
        spark.sql(sql)

## Initial data load into the empty dim_customer table

In [ ]:
from datetime import datetime
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DateType, BooleanType)

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("status", StringType(), True),
    StructField("effective_start_date", DateType(), True),
    StructField("effective_end_date", DateType(), True),
    StructField("current_flag", BooleanType(), True),
])

# Dana is seeded with a NULL email on purpose -- her row exercises the
# NULL -> value transition that a `!=` comparison would silently drop.
target_df = spark.createDataFrame([
    (1, "Alice", "alice@example.com", "active", datetime(2025, 1, 1), None, True),
    (2, "Bob", "bob@example.com", "active", datetime(2025, 1, 1), None, True),
    (4, "Dana", None, "active", datetime(2025, 1, 1), None, True),
], schema)

target_df.show()
target_df.write.mode("overwrite").format("delta").saveAsTable(TARGET_TABLE)

## New data including updates and inserts

In [ ]:
# One staging row per SCD2 path, so every branch is demonstrated:
#   Alice   -- tracked column changed  -> expire + insert a new current row
#   Bob     -- identical to target     -> no-op, stays on his original row
#   Charlie -- no current row          -> insert only
#   Dana    -- NULL -> value on email  -> expire + insert (dropped by `!=`)
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("status", StringType(), True),
])

source_df_changed = spark.createDataFrame([
    (1, "Alice", "alice@example.com", "inactive"),
    (2, "Bob", "bob@example.com", "active"),
    (3, "Charlie", "charlie@example.com", "active"),
    (4, "Dana", "dana@example.com", "active"),
], schema)

source_df_changed.createOrReplaceTempView(SOURCE_VIEW)
source_df_changed.show()

## Perform the SCD 2 load

Capture history when the following changes
 - name
 - email
 - status


In [ ]:
run_scd2_merge(
    source_table=SOURCE_VIEW,
    target_table=TARGET_TABLE,
    scd_keys="target.customer_id = source.customer_id",
    key_column="customer_id",
    tracked_columns=["name", "email", "status"],
    insert_columns=["customer_id", "name", "email", "status"],
    verbose=True,
)

In [ ]:
# Expected: Alice and Dana each have an expired row plus a new current row,
# Bob still has his single original current row, Charlie has one new row.
spark.sql(f"""
    SELECT * FROM {TARGET_TABLE}
    ORDER BY customer_id, effective_start_date, current_flag
""").show()